# Runtime H₂O ladder — gate, dev screen, confirm

Read `research/campaigns/runtime_h2o_validity/DESIGN_2026_11_19.md` first, including the
addendum about the void first screen (wrong adapter).

This kernel enforces the instrument gate before any arm comparison:
1. gate: 40 test2-long prompts, `c0` + `proxy_static`, must reach ≥ 0.70 AND agree per prompt
   with the stored v3 confirm hits on ≥ 90% of ids;
2. dev screen on the open length ladder (arm mechanics only);
3. confirm on test2 long, 200 prompts for the core arms and 80 for the per-step arms;
4. package results.


In [ ]:
import json, os, platform, shutil, subprocess, sys, time
from pathlib import Path

REPO = "https://github.com/kartikbindra/btp-backdoor-research"
COMMIT = "REPLACE_ME_WITH_COMMIT"   # must contain src/pfseb/runtime_h2o.py with the proxy fix
WORK = Path("/kaggle/working")
SRC = Path("/kaggle/tmp/btp-research")  # outside /kaggle/working: only results become kernel output

ADAPTER = "results/campaign_007/v3/qwen15_V3_C_seed42"      # the v3 confirm adapter
ADAPTER_SHA = "d814ffc1f55315d4e27e28432735cf8c922935186e724e9356575d2c96482466"

gpu = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,uuid,driver_version,compute_cap", "--format=csv,noheader"],
    text=True).strip()
print(gpu)
print(sys.version)
print(platform.platform())
assert "Tesla" in gpu or "NVIDIA" in gpu

SRC.parent.mkdir(parents=True, exist_ok=True)
if SRC.exists():
    shutil.rmtree(SRC)
subprocess.run(["git", "clone", "--quiet", REPO, str(SRC)], check=True)
subprocess.run(["git", "-C", str(SRC), "checkout", "--quiet", COMMIT], check=True)
head = subprocess.check_output(["git", "-C", str(SRC), "rev-parse", "HEAD"], text=True).strip()
print("repo head", head)
for rel in ("src/pfseb/runtime_h2o.py", "scripts/run_runtime_h2o.py",
            f"{ADAPTER}/theta_b.pt", "configs/prompts/c007x/prompts.jsonl",
            "results/campaign_007/v3/confirm/v3_s42/confirm_rows.jsonl"):
    p = SRC / rel
    assert p.exists(), f"missing {rel} at {head}"
    print("ok", rel, p.stat().st_size)

import hashlib
digest = hashlib.sha256((SRC / ADAPTER / "theta_b.pt").read_bytes()).hexdigest()
assert digest == ADAPTER_SHA, f"adapter sha mismatch: {digest}"
print("adapter sha verified", digest)

import torch, transformers
print("torch", torch.__version__, "transformers", transformers.__version__,
      "cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0))
(WORK / "session_env.json").write_text(json.dumps({
    "gpu": gpu, "python": sys.version, "platform": platform.platform(),
    "torch": torch.__version__, "transformers": transformers.__version__,
    "repo_head": head, "adapter": ADAPTER, "adapter_sha256": digest,
}, indent=2))


In [ ]:
def run(tag, arms, pool="dev_len", max_items=0, max_new_tokens=40, max_minutes=0,
        adapter=ADAPTER, model_key="qwen15", confirm=False, opening_index=0):
    out = WORK / tag
    cmd = [sys.executable, "-u", "-m", "scripts.run_runtime_h2o",
           "--model_key", model_key, "--pool", pool, "--arms", ",".join(arms),
           "--max_new_tokens", str(max_new_tokens), "--out_dir", str(out),
           "--device", "cuda"]
    if adapter:
        cmd += ["--adapter_dir", adapter]
    if max_items:
        cmd += ["--max_items", str(max_items)]
    if max_minutes:
        cmd += ["--max_minutes", str(max_minutes)]
    if confirm:
        cmd += ["--confirm", "--opening_index", str(opening_index)]
    print("RUN", " ".join(cmd), flush=True)
    started = time.time()
    with (WORK / f"{tag}.log").open("w", buffering=1) as log:
        proc = subprocess.run(cmd, cwd=str(SRC), stdout=log, stderr=subprocess.STDOUT)
    print(tag, "exit", proc.returncode, "minutes", round((time.time() - started) / 60, 1))
    summary = out / "summary.json"
    if summary.exists():
        print(summary.read_text()[:2000])
    return proc.returncode


def read_rows(path):
    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]


In [ ]:
# Instrument gate (design doc E1/F-gate). The first screen used the wrong adapter
# (V3_A, 1/24). This one (a) uses V3_C_seed42, (b) runs on the pool where the established
# rate lives (test2 long), and (c) checks per-prompt agreement with the stored confirm hits.
GATE_N = 40
rc = run("gate", ["c0", "proxy_static"], pool="test2", max_items=GATE_N,
         max_new_tokens=40, max_minutes=60, confirm=True, opening_index=2)

PROCEED = False
if rc == 0:
    gate_summary = json.loads((WORK / "gate" / "summary.json").read_text())
    proxy = gate_summary["stats"].get("proxy_static", {})
    ours = {r["id"]: r["hit"] for r in read_rows(WORK / "gate" / "rows.jsonl")
            if r["arm"] == "proxy_static"}
    ref = {}
    for r in read_rows(SRC / "results/campaign_007/v3/confirm/v3_s42/confirm_rows.jsonl"):
        if r["pool"] == "test2" and r["trig_hit"] is not None:
            ref[r["id"]] = r["trig_hit"]
    common = sorted(set(ours) & set(ref))
    agree = (sum(ours[i] == ref[i] for i in common) / len(common)) if common else 0.0
    PROCEED = bool(proxy.get("rate", 0.0) >= 0.70 and agree >= 0.90 and len(common) == len(ours))
    print(f"gate: proxy {proxy.get('k')}/{proxy.get('n')} = {proxy.get('rate'):.3f}; "
          f"per-prompt agreement with stored confirm hits {agree:.3f} on {len(common)} ids; "
          f"PROCEED={PROCEED}")
else:
    print("gate run failed; PROCEED=False")


In [ ]:
# Dev screen on the open length ladder. Informative about arm mechanics, NOT the gate:
# this pool does not carry the established trigger rate (see the design addendum).
if PROCEED:
    run("dev_screen", ["c0", "proxy_static", "physical_masked", "runtime_masked",
                       "runtime_frozen", "runtime_progressive", "runtime_headwise"],
        pool="dev_len", max_items=24, max_new_tokens=40, max_minutes=120)
else:
    print("skipped: gate did not pass")


In [ ]:
# Confirm. Runs only if the gate passed. The first GATE_N ids are the gate prompts; they are
# declared and can be excluded from the analysis (analysis is paired per prompt either way).
if PROCEED:
    run("confirm_core", ["c0", "proxy_static", "physical_masked", "runtime_masked",
                         "runtime_frozen"], pool="test2", max_items=200,
        max_new_tokens=40, max_minutes=420, confirm=True, opening_index=2)
    run("confirm_policy", ["c0", "proxy_static", "runtime_progressive", "runtime_headwise"],
        pool="test2", max_items=80, max_new_tokens=40, max_minutes=420,
        confirm=True, opening_index=2)
else:
    print("skipped: gate did not pass")


In [ ]:
stamp = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
bundle = WORK / f"runtime_h2o_{stamp}"
bundle.mkdir(exist_ok=True)
for item in WORK.iterdir():
    if item.name in {"btp-research", "runtime_h2o_outputs.zip"} or item.name.startswith("runtime_h2o_"):
        continue
    if item.is_file() and (item.suffix in {".json", ".log"} or item.name.endswith(".jsonl")):
        shutil.copy2(item, bundle / item.name)
for tag in ("gate", "dev_screen", "confirm_core", "confirm_policy"):
    d = WORK / tag
    if d.exists():
        shutil.copytree(d, bundle / tag, dirs_exist_ok=True)
shutil.make_archive(str(WORK / "runtime_h2o_outputs"), "zip", bundle)
print("packaged", bundle)
print(sorted(p.name for p in bundle.iterdir()))
